# Silver — espelho governado do bronze
A regra da casa, e ela não é negociável:
> **A silver é o espelho do bronze com governança aplicada.**
> Mesmo nome de tabela, mesmo grão, **mesma contagem de linhas**.
| | permitido na silver | proibido na silver |
|---|---|---|
| tipagem | ✅ string vira `TIMESTAMP`, `INT`, `DATE` | |
| legibilidade | ✅ quebrar timestamp em data e hora | |
| metadados | ✅ `COMMENT` em toda coluna, tags na tabela | |
| unificação | ✅ dois cadastros do mesmo assunto, com a origem por registro | |
| aritmética pura | ✅ `atraso = real - previsto` | |
| filtro / `WHERE` de negócio | | ❌ |
| `GROUP BY` / agregação | | ❌ |
| limiar, flag, classificação | | ❌ |

**Por quê?** Porque a silver precisa servir várias análises, e toda linha que ela
descarta é uma pergunta que ninguém mais vai conseguir fazer. Filtro fecha porta.
O teste para qualquer coluna nova: *isso embute uma decisão de negócio?*
`atraso_partida_min = partida_real - partida_prevista` é subtração — silver.
`partida_pontual = atraso <= 15` embute o número **15**, que é decisão de negócio
e muda por cliente — gold.

## 1. O que precisa ser consertado na tipagem
Antes de escrever o `CAST`, medir. Duas armadilhas escondidas no bronze:


In [0]:
display(spark.sql("""
    SELECT
        COUNT(*) AS linhas,
        SUM(CASE WHEN partida_real IS NULL THEN 1 ELSE 0 END) AS partida_real_null_de_verdade,
        SUM(CASE WHEN partida_real = 'null' THEN 1 ELSE 0 END) AS partida_real_null_string,
        SUM(CASE WHEN partida_prevista = 'null' THEN 1 ELSE 0 END) AS partida_prevista_string_null,
        SUM(CASE WHEN partida_prevista LIKE '%.%' THEN 1 ELSE 0 END) AS com_fracao_de_segundo
    FROM voebem.bronze.vra      
    """))

**Armadilha 1 — a ausência veio como a string `'null'`.** 
Quatro caracteres de texto. `WHERE partida_real IS NULL` devolve **zero** numa tabela onde 29 mil voos não têm
horário real. 

Correção: `nullif(coluna, 'null')` **antes** do cast.


**Armadilha 2 — dois formatos de timestamp no mesmo arquivo.** A maioria vem
`2026-01-27 19:45:00`, mas ~80 mil linhas vêm com fração de segundo de 9 casas.


Um `to_timestamp(col, 'yyyy-MM-dd HH:mm:ss')` fixo devolveria NULL para 8% da base,
em silêncio. O `try_cast(... AS TIMESTAMP)` aceita os dois formatos, e o `try_`
garante que um formato novo vire NULL em vez de derrubar o job.


Note que isso é **tipagem**, não limpeza de negócio: `'null'` é a forma como a fonte
escreve "ausente". Traduzir isso para `NULL` é dizer a mesma coisa no tipo certo.
Nenhuma linha sai.

## 2. `silver.vra` — o espelho

Repare no que **não** existe nesta query: nenhum `WHERE`, nenhum `GROUP BY`,
nenhum `DISTINCT`, nenhum `JOIN`. É um `SELECT` de projeção sobre o bronze inteiro.

E repare nas três colunas do fim: `atraso_partida_min`, `atraso_chegada_min` e
`minutos_recuperados`. São subtrações entre colunas da própria linha. Não têm
limiar, não classificam nada, não escondem número mágico — e, principalmente,
não impedem análise nenhuma. Por isso podem morar aqui.

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS voebem.silver
COMMENT 'Camada silver: espelho governado do bronze, com tipagem e metadados aplicados.'


In [0]:
spark.sql("""
    CREATE OR REPLACE TABLE voebem.silver.vra AS
    WITH tipado AS(
        SELECT
            icao_empresa,
            numero_voo,
            codigo_di,
            codigo_tipo_linha,
            icao_origem,
            icao_destino,
            try_cast(nullif(partida_prevista, 'null') AS TIMESTAMP) AS partida_prevista,
            try_cast(nullif(partida_real, 'null') AS TIMESTAMP) AS partida_real,
            try_cast(nullif(chegada_prevista, 'null') AS TIMESTAMP) AS chegada_prevista,
            try_cast(nullif(chegada_real, 'null') AS TIMESTAMP) AS chegada_real,
            situacao_voo,
            nullif(codigo_justificativa, 'N/A') AS codigo_justificativa,
            _arquivo_origem,
            _ingerido_em
        FROM voebem.bronze.vra
    )   
    SELECT
        icao_empresa,
        numero_voo,
        codigo_di,
        codigo_tipo_linha,
        icao_origem,
        icao_destino,
    
        partida_prevista,
        CAST(partida_prevista AS DATE) AS partida_prevista_data,
        date_format(partida_prevista, 'HH:mm') AS partida_prevista_hora,
        partida_real,
        CAST(partida_real AS DATE) AS partida_real_data,
        date_format(partida_real, 'HH:mm') AS partida_real_hora,
        chegada_prevista,
        CAST(chegada_prevista AS DATE) AS chegada_prevista_data,
        date_format(chegada_prevista, 'HH:mm') AS chegada_prevista_hora,
        chegada_real,
        CAST(chegada_real AS DATE) AS chegada_real_data,
        date_format(chegada_real, 'HH:mm') AS chegada_real_hora,

        situacao_voo,
        codigo_justificativa,

        -- aritmetica pura, opração apenas com dados da propria linha sem adicionar nada
        CAST(datediff(MINUTE, partida_prevista, partida_real) AS INT) AS atraso_partida_min,
        CAST(datediff(MINUTE, chegada_prevista, chegada_real) AS INT) AS atraso_chegada_min,
        CAST(datediff(MINUTE, partida_prevista, partida_real) - datediff(MINUTE, chegada_prevista, chegada_real) AS INT) AS minutos_recuperados,

        _arquivo_origem,
        _ingerido_em,
        current_timestamp() AS _transformado_em
    FROM tipado
    """)

print("silver.vra criada")